# Stage 5b — Per-Step Decode Latency Analysis

Measures per-decode-step latency for three methods on LongBench (qasper).

Unlike the aggregate latency in Stage 4, each token generation step is timed individually
using a short prefill (300 tokens) so the full sequence growth is visible over 6000 decode steps.

| Method | Description |
|--------|-------------|
| `full_cache` | Standard HuggingFace decoding, no eviction |
| `sliding_window` | Keep only most recent 256 tokens |
| `adaptive` | Our method: importance scoring + INT8 compression + budget-triggered eviction |

**No perplexity** — latency curve only.

## 1. Setup

In [ ]:
# Clone repo and install dependencies (run once per Colab session)
!git clone https://github.com/yanghao13111/adaptive-kv-cache.git 2>/dev/null || echo "Repo already cloned"
%cd adaptive-kv-cache
!git pull
# Downgrade datasets to support LongBench (datasets 4.0.0 dropped script support)
!pip install datasets==2.19.0 transformers accelerate -q

In [ ]:
# HuggingFace login — required to download Mistral-7B
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import sys
sys.path.insert(0, '/content/adaptive-kv-cache')

import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.eval.per_step import measure_per_step_latency
from src.adaptive.cache_manager import AdaptiveCacheManager

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Experiment Configuration

In [ ]:
MODEL_NAME = 'mistralai/Mistral-7B-v0.1'

BASE_CONFIG = {
    'dataset': 'longbench',
    'num_samples': 1,
    'prefill_tokens': 300,
    'max_new_tokens': 6000,
}

METHOD_CONFIGS = [
    {**BASE_CONFIG, 'method': 'full_cache'},
    {**BASE_CONFIG, 'method': 'sliding_window', 'window_size': 256},
]

ADAPTIVE_CONFIG = {
    **BASE_CONFIG,
    'method': 'adaptive',
    'method_kwargs': {
        'memory_budget_gb': 0.2,
        'recent_window': 256,
        'compress_dtype': 'int8',
        'sink_tokens': 4,
        'score_decay': 0.9,
    },
}

RESULTS_DIR = Path('experiments/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Mistral-7B GQA: 32 layers x 8 KV heads x 128 head_dim x 2 (k+v) x 2 bytes (FP16)
# budget 0.2 x 1024^3 bytes / 131,072 bytes_per_token = 1638 tokens
# With 300-token prefill -> eviction starts at decode step 1338
BUDGET_TOKENS = 1638
EVICTION_STEP = BUDGET_TOKENS - BASE_CONFIG['prefill_tokens']  # 1338

print(f"Methods: {[c['method'] for c in METHOD_CONFIGS]} + adaptive")
print(f"Prefill: {BASE_CONFIG['prefill_tokens']} tokens | Decode: {BASE_CONFIG['max_new_tokens']} steps")
print(f'Budget threshold: {BUDGET_TOKENS} tokens -> eviction at decode step ~{EVICTION_STEP}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()
print(f'Model loaded: {MODEL_NAME}')
print(f'Layers: {model.config.num_hidden_layers}')

## 3. LongBench Per-Step Results

Each method is run on the first `num_samples` documents from LongBench (qasper).
The first `prefill_tokens` tokens of each document are fed as prefill (not timed).
The following `max_new_tokens` decode steps are timed individually.

> **Note:** Step 0 of each run is excluded from the plot — it reflects a one-time CUDA warmup
> cost (kernel initialization after prefill) rather than steady-state decode latency.
>
> **Note:** The adaptive budget (0.2 GB) allows ~1638 tokens. With a 300-token prefill,
> eviction begins at decode step ~1338 and the cache stabilizes near the budget threshold.

In [ ]:
# Load and tokenize LongBench samples
dataset = load_dataset('THUDM/LongBench', 'qasper', split='test', trust_remote_code=True)
texts = [x['context'] for x in dataset if len(x['context'].split()) > 500]
texts = texts[:BASE_CONFIG['num_samples']]

prefill_ids_list = []
for i, text in enumerate(texts):
    ids = tokenizer(text, return_tensors='pt').input_ids[:, :BASE_CONFIG['prefill_tokens']].to(DEVICE)
    prefill_ids_list.append(ids)
    print(f'Sample {i+1}: {ids.shape[1]} tokens')

### 3a. Baselines (full_cache + sliding_window)

In [ ]:
baseline_records = []

for config in METHOD_CONFIGS:
    method = config['method']
    print(f"\n{'='*50}")
    print(f"Running: {method}")
    print(f"{'='*50}")

    for i, input_ids in enumerate(prefill_ids_list):
        print(f'  sample {i+1}/{len(prefill_ids_list)}...')
        steps = measure_per_step_latency(
            method, model, input_ids,
            max_new_tokens=config['max_new_tokens'],
            device=DEVICE,
            window_size=config.get('window_size', 256),
        )
        for r in steps:
            r['method'] = method
            r['sample_idx'] = i
        baseline_records.extend(steps)

    avg_lat = sum(r['latency_ms'] for r in baseline_records if r['method'] == method) / \
              sum(1 for r in baseline_records if r['method'] == method)
    print(f'  avg latency: {avg_lat:.2f} ms/token')

print('\nBaselines done.')

### 3b. Adaptive (memory_budget_gb=0.2)

In [ ]:
print(f"\n{'='*50}")
print("Running: adaptive")
print(f"{'='*50}")

adaptive_records = []
kwargs = ADAPTIVE_CONFIG['method_kwargs']

for i, input_ids in enumerate(prefill_ids_list):
    print(f'  sample {i+1}/{len(prefill_ids_list)}...')
    cache_manager = AdaptiveCacheManager(
        num_layers=model.config.num_hidden_layers,
        **kwargs,
    )
    steps = measure_per_step_latency(
        'adaptive', model, input_ids,
        max_new_tokens=ADAPTIVE_CONFIG['max_new_tokens'],
        device=DEVICE,
        cache_manager=cache_manager,
    )
    for r in steps:
        r['method'] = 'adaptive'
        r['sample_idx'] = i
    adaptive_records.extend(steps)

avg_lat = sum(r['latency_ms'] for r in adaptive_records) / len(adaptive_records)
print(f'  avg latency: {avg_lat:.2f} ms/token')
print('\nAdaptive done.')

## 4. Results

In [ ]:
df = pd.DataFrame(baseline_records + adaptive_records)
df = df[['method', 'sample_idx', 'step', 'cache_tokens', 'latency_ms']]
df['throughput_tps'] = (1000 / df['latency_ms']).round(2)

out_path = RESULTS_DIR / 'per_step_results.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows -> {out_path}')

# Exclude step 0 (CUDA warmup artifact) from summary
df_steady = df[df['step'] > 0]
print('\n=== Average (step 1 onwards) ===')
display(df_steady.groupby('method')[['latency_ms', 'throughput_tps', 'cache_tokens']].mean().round(2))

In [ ]:
FIG_DIR = Path('experiments/results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER  = ['full_cache', 'sliding_window', 'adaptive']
METHOD_LABELS = ['Full Cache', 'Sliding Window (256)', 'Adaptive (budget=0.2 GB)']
COLORS        = ['#2196F3', '#FF9800', '#4CAF50']
SMOOTH        = 30  # rolling average window to reduce per-step noise

# Exclude step 0 (warmup), average across samples
df_plot = df[df['step'] > 0].copy()
avg_lat   = df_plot.groupby(['method', 'step'])['latency_ms'].mean().reset_index()
avg_cache = df_plot.groupby(['method', 'step'])['cache_tokens'].mean().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for method, label, color in zip(METHOD_ORDER, METHOD_LABELS, COLORS):
    # --- Left: latency ---
    sub = avg_lat[avg_lat['method'] == method].sort_values('step')
    smoothed = sub['latency_ms'].rolling(SMOOTH, min_periods=1).mean()
    ax1.plot(sub['step'], smoothed, label=label, color=color, linewidth=1.8)

    # --- Right: cache tokens ---
    sub2 = avg_cache[avg_cache['method'] == method].sort_values('step')
    ax2.plot(sub2['step'], sub2['cache_tokens'], label=label, color=color, linewidth=1.8)

for ax in (ax1, ax2):
    ax.axvline(x=EVICTION_STEP, color='#4CAF50', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.text(EVICTION_STEP + 30, ax.get_ylim()[1] * 0.97,
            f'budget hit\n(step {EVICTION_STEP})',
            color='#388E3C', fontsize=8, va='top')

ax1.set_xlabel('Decode Step', fontsize=12)
ax1.set_ylabel('Latency (ms / token)', fontsize=12)
ax1.set_title('Per-Step Latency', fontsize=13)
ax1.legend(fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Decode Step', fontsize=12)
ax2.set_ylabel('KV Cache Size (tokens)', fontsize=12)
ax2.set_title('KV Cache Growth', fontsize=13)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

prefill = BASE_CONFIG['prefill_tokens']
fig.suptitle(
    f'Per-Step Decode Latency — LongBench ({prefill}-token prefill, step 0 excluded)',
    fontsize=13, y=1.02
)

plt.tight_layout()
out = FIG_DIR / 'fig5_per_step_latency.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')